# Notebook 8W — Context + Token RoBERTa (Women)

Women-discourse version. Subgroup identity is provided only inside `context_input_text`; there is no subgroup embedding and no FiLM.

Allowed subgroups are restricted to `women`, `men`, and `non_binary`.


In [7]:
import ast, json, random, itertools
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, confusion_matrix, classification_report
from scipy.spatial.distance import jensenshannon
from scipy.stats import entropy, pearsonr, spearmanr

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 220)

RANDOM_SEED = 42
MODEL_NAME = "roberta-base"
NUM_LABELS = 3

MAX_LENGTH = 384
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 1

EPOCHS = 8
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
GRAD_CLIP = 1.0
DROPOUT = 0.1

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = Path("/home/shayan/Distributional-Hate-Speech-Prediction/notebooks/data")
CONTEXT_PATH = Path("/home/shayan/Distributional-Hate-Speech-Prediction/notebooks/women_models/women_context_artifacts/women_context_mapped_examples.parquet")
OUTPUT_DIR = Path("context_token_women_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

ALLOWED_SUBGROUPS = ["women", "men", "non_binary"]

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)

print("Device:", DEVICE)
print("Context file:", CONTEXT_PATH)
print("Output directory:", OUTPUT_DIR.resolve())


Device: cuda
Context file: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/women_models/women_context_artifacts/women_context_mapped_examples.parquet
Output directory: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/women_models/context_token_women_outputs


## 1. Load data

In [8]:
context_df = pd.read_parquet(CONTEXT_PATH)

print("Raw context dataset:", context_df.shape)

context_df = context_df[context_df["subgroup"].isin(ALLOWED_SUBGROUPS)].copy().reset_index(drop=True)

print("Filtered context dataset:", context_df.shape)
print("Allowed subgroups:", ALLOWED_SUBGROUPS)
display(context_df.head(2))

required_columns = [
    "comment_id", "split", "subgroup", "text", "target_distribution",
    "target_majority_label", "context_input_text", "retrieved_article_titles",
    "retrieved_summaries",
]
missing = [c for c in required_columns if c not in context_df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Rows by split:")
display(context_df["split"].value_counts())

print("Rows by subgroup:")
display(context_df["subgroup"].value_counts())

print("Target majority distribution:")
display(context_df["target_majority_label"].value_counts(normalize=True).sort_index())

print("Unique context inputs:", context_df["context_input_text"].nunique(), "/", len(context_df))
print("Unique comments:", context_df["text"].nunique())

def safe_len(x):
    if isinstance(x, list): return len(x)
    if isinstance(x, np.ndarray): return len(x)
    if isinstance(x, str):
        try: return len(ast.literal_eval(x))
        except Exception: return np.nan
    return np.nan

print("Retrieved article count per row:")
display(context_df["retrieved_article_titles"].apply(safe_len).value_counts(dropna=False))

print("Example context:")
print("=" * 100)
print(context_df.iloc[0]["context_input_text"])


Raw context dataset: (7962, 16)
Filtered context dataset: (7962, 16)
Allowed subgroups: ['women', 'men', 'non_binary']


,experiment,comment_id,split,subgroup,subgroup_count,text,target_distribution,target_majority_label,target_expected_label,retrieved_article_titles,retrieved_page_urls,retrieved_similarities,retrieved_summaries,context_input_text,tweet_token_length,context_input_token_length
0,women,6,train,men,1,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),"[1.0, 0.0, 0.0]",0,0.0,[Male privilege],[https://en.wikipedia.org/wiki/Male_privilege],[0.10770449787378311],"[In patriarchal societies, males hold primary power and predominate in roles of leadership, moral authority, social privilege, and control of property, granting them economic, political, social, educational, and prac...",### COMMENT TO CLASSIFY\nFirst off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;)\n\n### ANNOTATOR GENDER\nmen\n\n### RETRIEVED BAC...,35,195
1,women,6,train,women,1,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),"[1.0, 0.0, 0.0]",0,0.0,[Misogyny],[https://en.wikipedia.org/wiki/Misogyny],[0.07197389751672745],"[Misogyny is a form of sexism that perpetuates women's subordinate status in patriarchal societies. It can manifest in obvious and subtle ways, including violence against women, sexual harassment, and exclusion from ...",### COMMENT TO CLASSIFY\nFirst off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;)\n\n### ANNOTATOR GENDER\nwomen\n\n### RETRIEVED B...,35,189


Rows by split:


split
train         5565
validation    1224
test          1173
Name: count, dtype: int64

Rows by subgroup:


...

Target majority distribution:


target_majority_label
0    0.677594
1    0.073851
2    0.248556
Name: proportion, dtype: float64

Unique context inputs: 7958 / 7962
Unique comments: 3951
Retrieved article count per row:


retrieved_article_titles
1    7962
Name: count, dtype: int64

Example context:
### COMMENT TO CLASSIFY
First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;)

### ANNOTATOR GENDER
men

### RETRIEVED BACKGROUND
Male privilege:
In patriarchal societies, males hold primary power and predominate in roles of leadership, moral authority, social privilege, and control of property, granting them economic, political, social, educational, and practical advantages over women. These privileges are often invisible to holders, leading them to attribute their status to individual merits rather than unearned advantages. The ideal masculine norm varies by society but is often associated with being white, heterosexual, stoic, wealthy, strong, tough, competitive, and autonomous. Men who deviate from this norm may not benefit fully from privilege, while those who conform to it are more likely to receive rewards and recognition.


## 2. Splits and tokenizer

In [9]:
train_df = context_df[context_df["split"] == "train"].reset_index(drop=True)
val_df = context_df[context_df["split"].isin(["validation", "val", "dev"])].reset_index(drop=True)
test_df = context_df[context_df["split"] == "test"].reset_index(drop=True)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

assert len(train_df) > 0 and len(val_df) > 0 and len(test_df) > 0

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def count_tokens(text):
    return len(tokenizer(text, truncation=False, add_special_tokens=True)["input_ids"])

if "context_input_token_length" not in context_df.columns:
    context_df["context_input_token_length"] = context_df["context_input_text"].apply(count_tokens)

display(pd.DataFrame([{
    "mean": context_df["context_input_token_length"].mean(),
    "median": context_df["context_input_token_length"].median(),
    "p95": context_df["context_input_token_length"].quantile(0.95),
    "max": context_df["context_input_token_length"].max(),
    "pct_over_384": float((context_df["context_input_token_length"] > 384).mean()),
}]))


Train: (5565, 16)
Val: (1224, 16)
Test: (1173, 16)


,mean,median,p95,max,pct_over_384
0,170.833459,170.0,223.0,296,0.0


## 3. Dataset and model

In [10]:
def parse_distribution(value):
    if isinstance(value, np.ndarray):
        return value.astype(float).tolist()
    if isinstance(value, list):
        return [float(x) for x in value]
    if isinstance(value, str):
        return [float(x) for x in ast.literal_eval(value)]
    raise TypeError(f"Unsupported distribution type: {type(value)}")


class ContextTokenDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(
            row["context_input_text"],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "target_distribution": torch.tensor(parse_distribution(row["target_distribution"]), dtype=torch.float),
        }


class ContextTokenRoBERTa(nn.Module):
    def __init__(self, model_name, num_labels=3, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_labels),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls)
        return {
            "logits": logits,
            "log_probs": torch.log_softmax(logits, dim=-1),
            "probs": torch.softmax(logits, dim=-1),
        }


train_loader = DataLoader(ContextTokenDataset(train_df, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(ContextTokenDataset(val_df, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(ContextTokenDataset(test_df, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)

batch = next(iter(train_loader))
print({k: v.shape for k, v in batch.items()})

model = ContextTokenRoBERTa(MODEL_NAME, NUM_LABELS, DROPOUT).to(DEVICE)

criterion = nn.KLDivLoss(reduction="batchmean")
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

num_update_steps_per_epoch = int(np.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS))
num_training_steps = num_update_steps_per_epoch * EPOCHS
num_warmup_steps = int(WARMUP_RATIO * num_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

print("Training steps:", num_training_steps)


{'input_ids': torch.Size([8, 384]), 'attention_mask': torch.Size([8, 384]), 'target_distribution': torch.Size([8, 3])}


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training steps: 5568


## 4. Metrics

In [11]:
EPS = 1e-12

def kl_divergence(y_true, y_pred):
    y_true = np.clip(y_true, EPS, 1.0)
    y_pred = np.clip(y_pred, EPS, 1.0)
    return np.sum(y_true * np.log(y_true / y_pred), axis=1)

def js_divergence(y_true, y_pred):
    return np.array([jensenshannon(t, p, base=2) ** 2 for t, p in zip(y_true, y_pred)])

def cross_entropy(y_true, y_pred):
    y_pred = np.clip(y_pred, EPS, 1.0)
    return -np.sum(y_true * np.log(y_pred), axis=1)

def expected_scores(distributions):
    return distributions @ np.arange(distributions.shape[1])

def entropy_values(distributions):
    return np.array([entropy(d, base=2) for d in distributions])

def compute_metrics(y_true, y_pred):
    true_labels = np.argmax(y_true, axis=1)
    pred_labels = np.argmax(y_pred, axis=1)
    true_entropy = entropy_values(y_true)
    pred_entropy = entropy_values(y_pred)

    out = {
        "kl_mean": float(kl_divergence(y_true, y_pred).mean()),
        "js_mean": float(js_divergence(y_true, y_pred).mean()),
        "cross_entropy_mean": float(cross_entropy(y_true, y_pred).mean()),
        "accuracy": float(accuracy_score(true_labels, pred_labels)),
        "macro_f1": float(f1_score(true_labels, pred_labels, average="macro", zero_division=0)),
        "expected_label_mae": float(mean_absolute_error(expected_scores(y_true), expected_scores(y_pred))),
    }
    if len(np.unique(true_entropy)) > 1 and len(np.unique(pred_entropy)) > 1:
        out["entropy_pearson"] = float(pearsonr(true_entropy, pred_entropy).statistic)
        out["entropy_spearman"] = float(spearmanr(true_entropy, pred_entropy).statistic)
    else:
        out["entropy_pearson"] = np.nan
        out["entropy_spearman"] = np.nan
    return out


## 5. Train

In [12]:
def run_epoch(model, dataloader, optimizer=None, scheduler=None):
    is_training = optimizer is not None
    model.train() if is_training else model.eval()

    total_loss = 0.0
    all_targets, all_predictions = [], []

    if is_training:
        optimizer.zero_grad()

    for step, batch in enumerate(dataloader):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        targets = batch["target_distribution"].to(DEVICE)

        with torch.set_grad_enabled(is_training):
            outputs = model(input_ids, attention_mask)
            raw_loss = criterion(outputs["log_probs"], targets)

            if is_training:
                (raw_loss / GRADIENT_ACCUMULATION_STEPS).backward()
                should_step = ((step + 1) % GRADIENT_ACCUMULATION_STEPS == 0) or ((step + 1) == len(dataloader))
                if should_step:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                    optimizer.step()
                    optimizer.zero_grad()
                    if scheduler is not None:
                        scheduler.step()

        total_loss += raw_loss.item() * input_ids.size(0)
        all_targets.append(targets.detach().cpu().numpy())
        all_predictions.append(outputs["probs"].detach().cpu().numpy())

    y_true = np.vstack(all_targets)
    y_pred = np.vstack(all_predictions)
    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = float(total_loss / len(dataloader.dataset))
    return metrics, y_true, y_pred


best_val_kl = float("inf")
best_model_path = OUTPUT_DIR / "context_token_best_model.pt"
history = []

for epoch in range(1, EPOCHS + 1):
    train_metrics, _, _ = run_epoch(model, train_loader, optimizer, scheduler)
    val_metrics, _, _ = run_epoch(model, val_loader)

    row = {"epoch": epoch, **{f"train_{k}": v for k, v in train_metrics.items()}, **{f"val_{k}": v for k, v in val_metrics.items()}}
    history.append(row)

    print("=" * 80)
    print(f"Epoch {epoch}/{EPOCHS}")
    print(f"Train KL: {train_metrics['kl_mean']:.4f} | Val KL: {val_metrics['kl_mean']:.4f}")
    print(f"Train JS: {train_metrics['js_mean']:.4f} | Val JS: {val_metrics['js_mean']:.4f}")
    print(f"Val Acc: {val_metrics['accuracy']:.4f} | Val Macro F1: {val_metrics['macro_f1']:.4f}")

    if val_metrics["kl_mean"] < best_val_kl:
        best_val_kl = val_metrics["kl_mean"]
        torch.save(model.state_dict(), best_model_path)
        print("Saved best model")

history_df = pd.DataFrame(history)
display(history_df)
history_df.to_csv(OUTPUT_DIR / "context_token_training_history.csv", index=False)


Epoch 1/8
Train KL: 0.7176 | Val KL: 0.6509
Train JS: 0.2838 | Val JS: 0.2607
Val Acc: 0.6993 | Val Macro F1: 0.4685
Saved best model
Epoch 2/8
Train KL: 0.6212 | Val KL: 0.6421
Train JS: 0.2349 | Val JS: 0.2342
Val Acc: 0.7198 | Val Macro F1: 0.4812
Saved best model
Epoch 3/8
Train KL: 0.5796 | Val KL: 0.6721
Train JS: 0.2181 | Val JS: 0.2407
Val Acc: 0.7108 | Val Macro F1: 0.4478
Epoch 4/8
Train KL: 0.5378 | Val KL: 0.6657
Train JS: 0.2008 | Val JS: 0.2384
Val Acc: 0.7181 | Val Macro F1: 0.4691
Epoch 5/8
Train KL: 0.4909 | Val KL: 0.7079
Train JS: 0.1856 | Val JS: 0.2304
Val Acc: 0.7157 | Val Macro F1: 0.4734
Epoch 6/8
Train KL: 0.4459 | Val KL: 0.7165
Train JS: 0.1673 | Val JS: 0.2311
Val Acc: 0.7198 | Val Macro F1: 0.4734
Epoch 7/8
Train KL: 0.4015 | Val KL: 0.7837
Train JS: 0.1526 | Val JS: 0.2374
Val Acc: 0.7181 | Val Macro F1: 0.4731
Epoch 8/8
Train KL: 0.3618 | Val KL: 0.8230
Train JS: 0.1369 | Val JS: 0.2384
Val Acc: 0.7157 | Val Macro F1: 0.4758


,epoch,train_kl_mean,train_js_mean,train_cross_entropy_mean,train_accuracy,train_macro_f1,train_expected_label_mae,train_entropy_pearson,train_entropy_spearman,train_loss,val_kl_mean,val_js_mean,val_cross_entropy_mean,val_accuracy,val_macro_f1,val_expected_label_mae,val_entropy_pearson,val_entropy_spearman,val_loss
0,1,0.717631,0.283822,0.798237,0.693082,0.410338,0.648127,0.075829,0.065965,0.717631,0.650936,0.260731,0.724066,0.699346,0.468524,0.589238,0.117069,0.123477,0.650936
1,2,0.621236,0.234899,0.701842,0.722911,0.469851,0.536066,0.138768,0.130482,0.621236,0.642103,0.234183,0.715233,0.719771,0.481223,0.521234,0.123159,0.118428,0.642103
2,3,0.579605,0.218107,0.660211,0.743037,0.487016,0.489041,0.177397,0.171779,0.579605,0.672062,0.240682,0.745192,0.710784,0.447847,0.522360,0.138603,0.134731,0.672062
3,4,0.537835,0.200780,0.618441,0.759748,0.498205,0.443332,0.201901,0.194832,0.537835,0.665726,0.238385,0.738856,0.718137,0.469115,0.507557,0.114383,0.108715,0.665726
4,5,0.490938,0.185633,0.571544,0.779515,0.515712,0.403715,0.230267,0.225642,0.490938,0.707873,0.230389,0.781003,0.715686,0.473440,0.492129,0.116854,0.102698,0.707873
5,6,0.445893,0.167310,0.526500,0.798922,0.534872,0.357811,0.252164,0.246437,0.445893,0.716489,0.231093,0.789619,0.719771,0.473371,0.488306,0.103509,0.096280,0.716489
6,7,0.401543,0.152556,0.482149,0.813118,0.584527,0.324855,0.285767,0.278845,0.401543,0.783731,0.237399,0.856861,0.718137,0.473114,0.497424,0.076670,0.080831,0.783731
7,8,0.361795,0.136885,0.442401,0.826235,0.627109,0.290347,0.301201,0.294051,0.361795,0.823018,0.238435,0.896148,0.715686,0.475794,0.489537,0.087064,0.095879,0.823018


## 6. Test and diagnostics

In [13]:
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
test_metrics, y_true_test, y_pred_test = run_epoch(model, test_loader)

display(test_metrics)

with open(OUTPUT_DIR / "context_token_test_metrics.json", "w") as f:
    json.dump(test_metrics, f, indent=2)

predictions_df = test_df.copy()
predictions_df["true_distribution"] = list(y_true_test)
predictions_df["pred_distribution"] = list(y_pred_test)
predictions_df["pred_majority_label"] = np.argmax(y_pred_test, axis=1)
predictions_df["pred_expected_label"] = expected_scores(y_pred_test)
predictions_df["pred_entropy"] = entropy_values(y_pred_test)
predictions_df["kl"] = kl_divergence(y_true_test, y_pred_test)
predictions_df["js"] = js_divergence(y_true_test, y_pred_test)
predictions_df["cross_entropy"] = cross_entropy(y_true_test, y_pred_test)

predictions_df.to_parquet(OUTPUT_DIR / "context_token_test_predictions.parquet", index=False)
predictions_df.to_csv(OUTPUT_DIR / "context_token_test_predictions.csv", index=False)

true_labels = np.argmax(y_true_test, axis=1)
pred_labels = np.argmax(y_pred_test, axis=1)

print("Confusion matrix:")
print(confusion_matrix(true_labels, pred_labels, labels=[0, 1, 2]))

print("\\nClassification report:")
print(classification_report(true_labels, pred_labels, labels=[0, 1, 2], zero_division=0))

print("\\nPredicted label distribution:")
display(pd.Series(pred_labels).value_counts(normalize=True).sort_index())

print("\\nTrue label distribution:")
display(pd.Series(true_labels).value_counts(normalize=True).sort_index())

print("\\nAverage predicted distribution:")
print(np.vstack(predictions_df["pred_distribution"].to_numpy()).mean(axis=0))

print("\\nMean KL by true majority label:")
display(
    predictions_df
    .groupby("target_majority_label")
    .agg(
        n=("comment_id", "count"),
        mean_kl=("kl", "mean"),
        mean_js=("js", "mean"),
        mean_target_entropy=("target_distribution", lambda x: np.mean([entropy(parse_distribution(v), base=2) for v in x])),
        mean_pred_entropy=("pred_entropy", "mean"),
    )
)

print("Average predicted distribution by true majority label:")
for label in [0, 1, 2]:
    subset = predictions_df[predictions_df["target_majority_label"] == label]
    avg_pred = np.vstack(subset["pred_distribution"].to_numpy()).mean(axis=0)
    print(label, avg_pred)

print("\\nPerformance by subgroup:")
subgroup_rows = []
for subgroup, group in predictions_df.groupby("subgroup"):
    y_true = np.vstack(group["true_distribution"].to_numpy())
    y_pred = np.vstack(group["pred_distribution"].to_numpy())
    subgroup_rows.append({"subgroup": subgroup, "n": len(group), **compute_metrics(y_true, y_pred)})

subgroup_metrics_df = pd.DataFrame(subgroup_rows).sort_values("kl_mean")
display(subgroup_metrics_df)
subgroup_metrics_df.to_csv(OUTPUT_DIR / "context_token_subgroup_metrics.csv", index=False)


/tmp/ipykernel_198138/1013887499.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))


{'kl_mean': 0.6798139810562134,
 'js_mean': 0.24370340982974747,
 'cross_entropy_mean': 0.7632770538330078,
 'accuracy': 0.6973572037510657,
 'macro_f1': 0.45881008087347624,
 'expected_label_mae': 0.5596660129584607,
 'entropy_pearson': 0.13448516902842442,
 'entropy_spearman': 0.12051758709050223,
 'loss': 0.6798139735738741}

Confusion matrix:
[[602   0 183]
 [ 49   0  37]
 [ 86   0 216]]
\nClassification report:
              precision    recall  f1-score   support

           0       0.82      0.77      0.79       785
           1       0.00      0.00      0.00        86
           2       0.50      0.72      0.59       302

    accuracy                           0.70      1173
   macro avg       0.44      0.49      0.46      1173
weighted avg       0.67      0.70      0.68      1173

\nPredicted label distribution:


0    0.628303
2    0.371697
Name: proportion, dtype: float64

\nTrue label distribution:


0    0.669224
1    0.073316
2    0.257460
Name: proportion, dtype: float64

\nAverage predicted distribution:
[0.66559595 0.05364672 0.28075793]
\nMean KL by true majority label:


,n,mean_kl,mean_js,mean_target_entropy,mean_pred_entropy
target_majority_label,,,,,
0,785,0.346985,0.142039,0.142549,0.742589
1,86,2.584393,0.758118,0.197674,0.936875
2,302,1.002584,0.361474,0.040868,1.075404


Average predicted distribution by true majority label:
0 [0.7516252  0.04945444 0.19892053]
1 [0.615603   0.05921541 0.32518137]
2 [0.45621207 0.06295799 0.48082995]
\nPerformance by subgroup:


,subgroup,n,kl_mean,js_mean,cross_entropy_mean,accuracy,macro_f1,expected_label_mae,entropy_pearson,entropy_spearman
1,non_binary,16,0.619833,0.225861,0.619833,0.750000,0.494949,0.521688,NaN,NaN
0,men,576,0.672209,0.239092,0.764803,0.680556,0.433711,0.554514,0.173042,0.144307
2,women,581,0.689005,0.248766,0.765714,0.712565,0.479545,0.565820,0.094924,0.094244


## 7. Counterfactual subgroup sensitivity

In [14]:
@torch.no_grad()
def predict_distribution(context_text):
    model.eval()
    enc = tokenizer(
        context_text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    outputs = model(enc["input_ids"].to(DEVICE), enc["attention_mask"].to(DEVICE))
    return outputs["probs"].detach().cpu().numpy()[0]


subgroups = sorted(context_df["subgroup"].unique())
context_lookup = {(row["comment_id"], row["subgroup"]): row["context_input_text"] for _, row in context_df.iterrows()}
unique_test_comments = test_df[["comment_id", "text"]].drop_duplicates("comment_id").reset_index(drop=True)

counterfactual_rows = []
for _, row in unique_test_comments.iterrows():
    comment_id = row["comment_id"]
    available_subgroups = [s for s in subgroups if (comment_id, s) in context_lookup]

    if len(available_subgroups) < 2:
        continue

    preds_by_group = {s: predict_distribution(context_lookup[(comment_id, s)]) for s in available_subgroups}

    pairwise_js = []
    for group_a, group_b in itertools.combinations(available_subgroups, 2):
        pairwise_js.append(jensenshannon(preds_by_group[group_a], preds_by_group[group_b], base=2) ** 2)

    counterfactual_rows.append({
        "comment_id": comment_id,
        "text": row["text"],
        "n_available_subgroups": len(available_subgroups),
        "mean_pairwise_js": float(np.mean(pairwise_js)),
        "max_pairwise_js": float(np.max(pairwise_js)),
    })

cf_df = pd.DataFrame(counterfactual_rows)

print("Counterfactual rows:", cf_df.shape)
print("Mean counterfactual JS:", cf_df["mean_pairwise_js"].mean())
print("Median counterfactual JS:", cf_df["mean_pairwise_js"].median())
print("Mean max-pairwise JS:", cf_df["max_pairwise_js"].mean())
print("Median max-pairwise JS:", cf_df["max_pairwise_js"].median())

display(cf_df.sort_values("max_pairwise_js", ascending=False).head(30))
cf_df.to_csv(OUTPUT_DIR / "context_token_counterfactual_js.csv", index=False)


Counterfactual rows: (582, 5)
Mean counterfactual JS: 0.003924582140302166
Median counterfactual JS: 0.00022571312130560776
Mean max-pairwise JS: 0.003931922616713469
Median max-pairwise JS: 0.00022778900564610026


,comment_id,text,n_available_subgroups,mean_pairwise_js,max_pairwise_js
182,12292,Exploded on that slut,2,0.378625,0.378625
540,44397,That's not accurate at all. He slut shamed her,2,0.173388,0.173388
455,35197,Where the single friendly lesbians at ? Not the ones wit gfs in whole relationships an pregnant tf,2,0.075972,0.075972
62,3696,What a hot little fuck puppet she is! Would fuck her in the ass all night long!,2,0.070023,0.070023
559,47822,Survey says women are emotional lunatics.,2,0.065952,0.065952
247,17937,So much to dislike about this without even opening the article. Cardi B? Disgusting talentless trash. Breitbart? Rightwing filth. Ignoring the white supremacist in El Paso who actually did target minorities? I want t...,2,0.060877,0.060877
547,45258,"These 4 women are hate mongers and racists, they are actually bullying the dems. Members ,they are scared of them, these women need to be spanked! Maga",2,0.057385,0.057385
545,44867,WOW I wish these Hos would just go to hell already. Worthless pile of rebellious trash these women are.,2,0.049076,0.049076
151,10655,I also don't care for calling a woman a b****. It's an inherently sexist term smh man,2,0.047764,0.047764
423,32956,Bitches too!,2,0.047614,0.047614


## 8. Men vs women and true-vs-model disagreement


In [15]:
def pairwise_counterfactual_js(group_a, group_b):
    rows = []
    unique_comments = test_df[["comment_id", "text"]].drop_duplicates("comment_id").reset_index(drop=True)

    for _, row in unique_comments.iterrows():
        comment_id = row["comment_id"]
        if (comment_id, group_a) not in context_lookup or (comment_id, group_b) not in context_lookup:
            continue

        context_a = context_lookup[(comment_id, group_a)]
        context_b = context_lookup[(comment_id, group_b)]

        pred_a = predict_distribution(context_a)
        pred_b = predict_distribution(context_b)

        rows.append({
            "comment_id": comment_id,
            "text": row["text"],
            "group_a": group_a,
            "group_b": group_b,
            "js": float(jensenshannon(pred_a, pred_b, base=2) ** 2),
            f"pred_{group_a}": pred_a,
            f"pred_{group_b}": pred_b,
            f"context_{group_a}": context_a,
            f"context_{group_b}": context_b,
        })

    return pd.DataFrame(rows)


def valid_dist(x):
    try:
        arr = np.array(parse_distribution(x), dtype=float)
    except Exception:
        return False
    return arr.shape[0] == NUM_LABELS and not np.isnan(arr).any() and arr.sum() > 0


def true_pairwise_disagreement_from_context_df(full_df, group_a, group_b):
    rows = []

    for comment_id, group in full_df.groupby("comment_id"):
        if group_a not in set(group["subgroup"]) or group_b not in set(group["subgroup"]):
            continue

        row_a = group[group["subgroup"] == group_a].iloc[0]
        row_b = group[group["subgroup"] == group_b].iloc[0]

        dist_a = parse_distribution(row_a["target_distribution"])
        dist_b = parse_distribution(row_b["target_distribution"])

        if not valid_dist(dist_a) or not valid_dist(dist_b):
            continue

        dist_a = np.array(dist_a, dtype=float)
        dist_b = np.array(dist_b, dtype=float)

        rows.append({
            "comment_id": comment_id,
            "text": row_a["text"],
            "true_js": float(jensenshannon(dist_a, dist_b, base=2) ** 2),
            f"{group_a}_true_dist": dist_a,
            f"{group_b}_true_dist": dist_b,
            f"{group_a}_true_label": int(np.argmax(dist_a)),
            f"{group_b}_true_label": int(np.argmax(dist_b)),
        })

    return pd.DataFrame(rows)


if "men" in subgroups and "women" in subgroups:
    men_women_df = pairwise_counterfactual_js("men", "women")

    print("Men vs women rows:", men_women_df.shape)
    print("Men vs women mean JS:", men_women_df["js"].mean())
    print("Men vs women median JS:", men_women_df["js"].median())
    display(men_women_df.sort_values("js", ascending=False).head(30))
    men_women_df.to_csv(OUTPUT_DIR / "context_token_men_vs_women_counterfactual.csv", index=False)

    true_df = true_pairwise_disagreement_from_context_df(context_df, "men", "women")

    comparison_df = true_df.merge(
        men_women_df[["comment_id", "js", "pred_men", "pred_women"]],
        on="comment_id",
        how="inner",
    ).rename(columns={"js": "model_js"})

    nonzero = comparison_df[comparison_df["true_js"] > 0].copy()
    nonzero["capture_ratio"] = nonzero["model_js"] / nonzero["true_js"]

    print("N true men/women overlap:", len(true_df))
    print("N comparison:", len(comparison_df))
    print("Mean true JS:", comparison_df["true_js"].mean())
    print("Mean model JS:", comparison_df["model_js"].mean())
    if len(nonzero) > 0:
        print("Mean capture ratio, true_js > 0:", nonzero["capture_ratio"].mean())
        print("Median capture ratio, true_js > 0:", nonzero["capture_ratio"].median())

    comparison_df["label_pair"] = (
        comparison_df["men_true_label"].astype(str)
        + "-"
        + comparison_df["women_true_label"].astype(str)
    )

    display(
        comparison_df
        .groupby("label_pair")
        .agg(n=("comment_id", "count"), mean_true_js=("true_js", "mean"), mean_model_js=("model_js", "mean"))
        .sort_values("mean_true_js", ascending=False)
    )

    comparison_df.to_csv(OUTPUT_DIR / "context_token_true_vs_model_men_women.csv", index=False)

    if len(men_women_df) > 0:
        top_row = men_women_df.sort_values("js", ascending=False).iloc[0]
        print("\nHighest men/women context effect")
        print("Comment:")
        print(top_row["text"])
        print("\nPrediction men:")
        print(top_row["pred_men"])
        print("\nPrediction women:")
        print(top_row["pred_women"])
        print("\nContext men:")
        print(top_row["context_men"])
        print("\nContext women:")
        print(top_row["context_women"])
else:
    print("Skipping men vs women analysis because one or both subgroups are missing.")


Men vs women rows: (575, 9)
Men vs women mean JS: 0.003865131423501951
Men vs women median JS: 0.0002248128996174977


,comment_id,text,group_a,group_b,js,pred_men,pred_women,context_men,context_women
181,12292,Exploded on that slut,men,women,0.378625,"[0.925823, 0.030853232, 0.043323785]","[0.27120948, 0.05831945, 0.6704712]","### COMMENT TO CLASSIFY\nExploded on that slut\n\n### ANNOTATOR GENDER\nmen\n\n### RETRIEVED BACKGROUND\nMale privilege:\nIn patriarchal societies, males hold primary power and predominate in roles of leadership, mor...",### COMMENT TO CLASSIFY\nExploded on that slut\n\n### ANNOTATOR GENDER\nwomen\n\n### RETRIEVED BACKGROUND\nViolence against women:\nViolence against women is a manifestation of historically unequal power relations be...
535,44397,That's not accurate at all. He slut shamed her,men,women,0.173388,"[0.74508244, 0.07132597, 0.18359159]","[0.29220715, 0.05950029, 0.6482926]","### COMMENT TO CLASSIFY\nThat's not accurate at all. He slut shamed her\n\n### ANNOTATOR GENDER\nmen\n\n### RETRIEVED BACKGROUND\nMale privilege:\nIn patriarchal societies, males hold primary power and predominate in...",### COMMENT TO CLASSIFY\nThat's not accurate at all. He slut shamed her\n\n### ANNOTATOR GENDER\nwomen\n\n### RETRIEVED BACKGROUND\nMisogyny:\nMisogyny is a form of sexism that perpetuates women's subordinate status ...
450,35197,Where the single friendly lesbians at ? Not the ones wit gfs in whole relationships an pregnant tf,men,women,0.075972,"[0.8112318, 0.060143847, 0.12862444]","[0.51720977, 0.09270209, 0.39008817]",### COMMENT TO CLASSIFY\nWhere the single friendly lesbians at ? Not the ones wit gfs in whole relationships an pregnant tf\n\n### ANNOTATOR GENDER\nmen\n\n### RETRIEVED BACKGROUND\nGender role:\nGender roles are soc...,### COMMENT TO CLASSIFY\nWhere the single friendly lesbians at ? Not the ones wit gfs in whole relationships an pregnant tf\n\n### ANNOTATOR GENDER\nwomen\n\n### RETRIEVED BACKGROUND\nMisogyny:\nMisogyny is a form of...
61,3696,What a hot little fuck puppet she is! Would fuck her in the ass all night long!,men,women,0.070023,"[0.7186626, 0.0832941, 0.19804327]","[0.4313027, 0.08788899, 0.48080826]","### COMMENT TO CLASSIFY\nWhat a hot little fuck puppet she is! Would fuck her in the ass all night long!\n\n### ANNOTATOR GENDER\nmen\n\n### RETRIEVED BACKGROUND\nMasculinity:\nIn Western cultures, traditional mascul...",### COMMENT TO CLASSIFY\nWhat a hot little fuck puppet she is! Would fuck her in the ass all night long!\n\n### ANNOTATOR GENDER\nwomen\n\n### RETRIEVED BACKGROUND\nViolence against women:\nViolence against women is ...
553,47822,Survey says women are emotional lunatics.,men,women,0.065952,"[0.80877346, 0.059399202, 0.1318274]","[0.53593516, 0.09021692, 0.3738479]","### COMMENT TO CLASSIFY\nSurvey says women are emotional lunatics.\n\n### ANNOTATOR GENDER\nmen\n\n### RETRIEVED BACKGROUND\nGender role:\nGender roles are social norms that vary across cultures, influencing a wide r...",### COMMENT TO CLASSIFY\nSurvey says women are emotional lunatics.\n\n### ANNOTATOR GENDER\nwomen\n\n### RETRIEVED BACKGROUND\nMisogyny:\nMisogyny is a form of sexism that perpetuates women's subordinate status in pa...
243,17937,So much to dislike about this without even opening the article. Cardi B? Disgusting talentless trash. Breitbart? Rightwing filth. Ignoring the white supremacist in El Paso who actually did target minorities? I want t...,men,women,0.060877,"[0.73137754, 0.071799845, 0.19682264]","[0.45821884, 0.08896139, 0.4528197]",### COMMENT TO CLASSIFY\nSo much to dislike about this without even opening the article. Cardi B? Disgusting talentless trash. Breitbart? Rightwing filth. Ignoring the white supremacist in El Paso who actually did ta...,### COMMENT TO CLASSIFY\nSo much to dislike about this without even opening the article. Cardi B? Disgusting talentless trash. Breitbart? Rightwing filth. Ignoring the white supremacist in El Paso who actually did ta...
540,44867,WOW I wish these Hos would just go to hell already. Worthless pile of rebellious trash these women are.,men,women,0.

N true men/women overlap: 3869
N comparison: 575
Mean true JS: 0.34650682259322935
Mean model JS: 0.003865131423501951
Mean capture ratio, true_js > 0: 0.01491149595046095
Median capture ratio, true_js > 0: 0.00040808903142176535


,n,mean_true_js,mean_model_js
label_pair,,,
1-0,21,1.000000,0.001356
2-1,7,1.000000,0.016335
0-1,32,0.962852,0.003479
2-0,50,0.865213,0.002191
0-2,84,0.828284,0.003055
1-2,12,0.827820,0.002044
0-0,291,0.055885,0.003579
1-1,6,0.051880,0.007138
2-2,72,0.015082,0.006850



Highest men/women context effect
Comment:
Exploded on that slut

Prediction men:
[0.925823   0.03085323 0.04332379]

Prediction women:
[0.27120948 0.05831945 0.6704712 ]

Context men:
### COMMENT TO CLASSIFY
Exploded on that slut

### ANNOTATOR GENDER
men

### RETRIEVED BACKGROUND
Male privilege:
In patriarchal societies, males hold primary power and predominate in roles of leadership, moral authority, social privilege, and control of property, granting them economic, political, social, educational, and practical advantages over women. These privileges are often invisible to holders, leading them to attribute their status to individual merits rather than unearned advantages. The ideal masculine norm varies by society but is often associated with being white, heterosexual, stoic, wealthy, strong, tough, competitive, and autonomous. Men who deviate from this norm may not benefit fully from privilege, while those who conform to it are more likely to receive rewards and recognition.

Cont

## Interpretation

This women-discourse model tests whether putting subgroup identity and retrieved context directly through RoBERTa attention is enough, without subgroup embeddings or FiLM.

Allowed subgroups are restricted to `women`, `men`, and `non_binary`; `prefer_not_to_say` and `self_describe` are excluded.

Compare against:

- Token baseline without context
- Embedding baseline without context
- Context + Embedding
- Context + Strong FiLM
